# 01 — Esplorazione del dataset Ames Housing

## Obiettivi didattici

1. Comprendere la struttura del dataset (2930 osservazioni, 82 variabili).
2. Distinguere tipi di variabili: numeriche, categoriche nominali, categoriche ordinali, identificatori.
3. Esaminare la **distribuzione del target** `SalePrice` e identificare la skewness, motivando la trasformazione logaritmica.
4. Analizzare i **valori mancanti** distinguendo fra missing strutturali (es. `PoolQC=NaN` ⇒ niente piscina) e missing veri.
5. Identificare gli **outlier** raccomandati da De Cock (2011).

## Riferimenti

- De Cock, D. (2011). *Ames, Iowa: Alternative to the Boston Housing Data...* Journal of Statistics Education 19(3).
- Documentazione delle 82 variabili: `data/raw/DataDocumentation.txt`.


In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

from ames_pipeline.data import load_raw
from ames_pipeline.wrangling import (
    fill_structural_missing,
    remove_grliv_area_outliers,
    NA_AS_NONE_CATEGORICAL,
    NA_AS_ZERO_NUMERIC,
)
from ames_pipeline.preprocessing import infer_column_groups, ORDINAL_CATEGORIES_MAP


## Caricamento

Il dataset è il file originale `AmesHousing.txt` (tab-separated) scaricato dalla pagina ufficiale del Journal of Statistics Education. La funzione `load_raw()` esegue:

1. download dal mirror JSE se assente,
2. validazione SHA-256 (per riproducibilità),
3. normalizzazione dei nomi colonna (rimuove spazi e `/`).


In [ ]:
df = load_raw()
print(f'Shape: {df.shape}')
print(f'Target: SalePrice  (min=${df.SalePrice.min():,}, max=${df.SalePrice.max():,})')
df.head(3)

## Composizione delle variabili

Suddividiamo le colonne in tre famiglie (escludendo gli identificatori `Order`, `PID` che vanno droppati subito):

- **Numeriche** continue/discrete (`LotArea`, `GrLivArea`, `OverallQual`, …)
- **Ordinali** (qualità decrescente: Po<Fa<TA<Gd<Ex)
- **Nominali** (`Neighborhood`, `HouseStyle`, …)

*Nota didattica:* la classificazione qui è basata sulla documentazione del dataset, non su euristica automatica. Inferire 'numerica vs categorica' dal solo dtype porta errori (es. `OverallQual` è int ma logicamente ordinale).


In [ ]:
groups = infer_column_groups(df.drop(columns=['Order','PID','SalePrice']))
for k, cols in groups.items():
    print(f'{k:9s}: {len(cols):2d} colonne')
print()
print('Ordinali con ordine semantico predefinito:')
for col in groups['ordinal']:
    print(f'  {col:15s} → {ORDINAL_CATEGORIES_MAP.get(col)}')


## Distribuzione del target `SalePrice`

Il prezzo è **fortemente asimmetrico a destra** (long tail di case di lusso). Conseguenza: i modelli che minimizzano l'errore quadratico (RMSE) sono dominati dalle case costose. Trasformiamo il target con **log1p** per:

- avvicinarlo a una gaussiana (assunzione comoda per Ridge),
- rendere l'errore relativo più uniforme su tutte le fasce di prezzo,
- usare RMSE-log come metrica (lo stesso scoring di Kaggle Ames Competition).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df.SalePrice, bins=60, edgecolor='black')
axes[0].set(title='SalePrice (scala originale)', xlabel='$', ylabel='count')
axes[0].ticklabel_format(style='plain', axis='x')
axes[1].hist(np.log1p(df.SalePrice), bins=60, edgecolor='black', color='C1')
axes[1].set(title='log1p(SalePrice) — quasi-normale', xlabel='log $')
fig.tight_layout(); plt.show()

print(f'Skewness originale  : {df.SalePrice.skew():.3f}')
print(f'Skewness log1p      : {np.log1p(df.SalePrice).skew():.3f}')


## Analisi dei valori mancanti

**Punto critico**: in Ames Housing, il `NaN` di una colonna come `PoolQC` non è un dato mancante — significa **"casa senza piscina"**. La documentazione di De Cock lo specifica per ~16 colonne. Imputare la moda/mediana qui sarebbe sbagliato: introdurrebbe rumore. Mappiamo invece questi NaN a una categoria 'None' (per categoriche) o `0` (per numeriche associate). Solo `LotFrontage` (e residui) hanno mancanti veri da imputare.

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_df = pd.DataFrame({
    'col': missing.index,
    'missing': missing.values,
    'pct': (missing.values / len(df) * 100).round(1),
    'tipo': [
        'strutturale (NA→None)' if c in NA_AS_NONE_CATEGORICAL else
        'strutturale (NA→0)'    if c in NA_AS_ZERO_NUMERIC else
        'vero (impute)'
        for c in missing.index
    ],
})
missing_df

In [ ]:
df_clean = fill_structural_missing(df)
still_missing = df_clean.isna().sum()
still_missing = still_missing[still_missing > 0]
print(f'Dopo fill_structural_missing rimangono mancanti su:')
print(still_missing.to_string())
print('\n→ Solo veri mancanti (verranno imputati con SimpleImputer dentro la pipeline).')


## Outlier raccomandati da De Cock

Il paper originale segnala 5 case con `GrLivArea > 4000 ft²` e prezzo anomalmente basso. Sono vendite particolari (parziali, fra parenti, liquidazioni) che rovinano i modelli lineari. Vanno rimosse PRIMA dello split.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df.GrLivArea, df.SalePrice, alpha=0.4, s=15)
outliers = df[(df.GrLivArea > 4000) & (df.SalePrice < 300_000)]
ax.scatter(outliers.GrLivArea, outliers.SalePrice, color='red', s=80,
           label=f'outlier ({len(outliers)})', zorder=5)
ax.set(xlabel='GrLivArea (ft²)', ylabel='SalePrice ($)',
       title='Outlier GrLivArea segnalati da De Cock')
ax.legend(); plt.show()


## Top correlazioni con il target

Le correlazioni di Pearson catturano relazioni lineari. Le più informative guideranno la baseline lineare; per i modelli non lineari (RF/XGB) tutte le feature contribuiscono.

In [ ]:
df_num = df_clean.select_dtypes(include='number').drop(columns=['Order','PID'])
corr = df_num.corr(numeric_only=True)['SalePrice'].sort_values(ascending=False)
top10 = corr.head(11).iloc[1:]  # esclude SalePrice stesso
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top10.index[::-1], top10.values[::-1])
ax.set(title='Top-10 correlazioni di Pearson con SalePrice',
       xlabel='Pearson r')
for i, v in enumerate(top10.values[::-1]):
    ax.text(v, i, f' {v:.3f}', va='center')
plt.tight_layout(); plt.show()


## Conclusioni dell'EDA e implicazioni per il modeling

| Osservazione | Implicazione |
|---|---|
| Target skewed | Usare `log1p(SalePrice)` come variabile dipendente. |
| 16+ colonne con NaN strutturali | Pre-fill **prima** dello split (no leakage: è semantica). |
| `LotFrontage` ha 490 missing veri | Imputazione **dentro** la pipeline (mediana sul train). |
| 5 outlier `GrLivArea` | Rimozione raccomandata da De Cock prima dello split. |
| 23 ordinali con ordinamento Po<Fa<TA<Gd<Ex | `OrdinalEncoder` con `categories=...` esplicite. |
| `OverallQual` r=0.80 con prezzo | Feature dominante; baseline lineare già forte. |
| Cardinalità di `Neighborhood`=28 | OneHotEncoder con `min_frequency=2`. |

→ Procedi al notebook **02_preprocessing_features**.
